## What is indexing and how do documents get indexed?

Indexing is the process of organizing documents before a user searches them. Its purpose is to avoid reading every document from beginning to end for every query. During indexing, a search system usually gives each document an ID, processes its text, extracts its terms, and stores a structure that connects those terms to the documents in which they occur.

### Document-term frequency matrix

A document-term frequency matrix has one row for every document and one column for every term in the collection. Each cell stores the number of times that term occurs in that document. It preserves useful frequency information, but a realistic vocabulary can contain hundreds of thousands of terms, most cells are zero, and adding new terms requires new columns. Searching a large dense matrix would therefore waste both memory and computation.

### Document-term incidence matrix

A document-term incidence matrix has the same document-by-term shape, but each cell contains only `1` if the term occurs in the document and `0` otherwise. This is sufficient for Boolean questions such as “which documents contain this word?”, but it discards how many times the term occurs. It is also extremely sparse and has the same large-matrix problem as the frequency matrix.

### Inverted index

An inverted index reverses the document-term view. Instead of storing every possible term position for every document, it stores each term once and associates it with a **postings list** containing the IDs of documents in which that term occurs. A posting can also store the term frequency and positions inside the document.

For example, instead of a mostly empty column for `movie`, an inverted index might store `movie → [(2, 4), (7, 1)]`, meaning that the term occurs four times in document 2 and once in document 7. Terms that do not occur in a document require no entry. This directly represents only the non-zero information, reduces wasted storage, and lets the search engine retrieve a term's matching documents immediately. Multi-term queries can then be processed by intersecting or combining postings lists rather than scanning every document.

## What is TF-IDF?

**Term frequency (TF)** measures how often a term appears in one document. A normalized form is

$$TF(t,d)=\frac{\text{count of term }t\text{ in document }d}{\text{total terms in document }d}.$$

Regular term frequency assumes that a word is important when it appears many times in a document. Its problem is that words which occur frequently throughout the entire collection can receive high scores even though they do little to distinguish one document from another. For example, a common word may occur many times in a movie review without revealing what makes that review relevant to a query.

**Document frequency (DF)** is the number of documents that contain a term at least once. Repeating a term many times inside one document does not increase its DF. If $N$ is the total number of documents, inverse document frequency is

$$IDF(t)=\log\left(\frac{N}{DF(t)}\right).$$

The fraction is inverted so that the value decreases as document frequency increases. A rare term has a small DF, so $N/DF$ and its IDF are large. A term found in nearly every document has a large DF, so its IDF approaches zero. The logarithm compresses the ratio: without it, an extremely rare term could receive a disproportionately large weight and dominate the score. It also makes changes in rarity more gradual.

TF-IDF combines the two quantities:

$$TFIDF(t,d)=TF(t,d)\times IDF(t).$$

Multiplication rewards a term for appearing frequently in the current document while penalizing it for appearing in many documents. Therefore, a term receives a high score when it is frequent in this document but relatively rare across the collection. This makes TF-IDF more informative than regular term frequency alone.

In [2]:
from collections import Counter
from pathlib import Path
import math
import sys

import pandas as pd


def term_frequency(term, document_tokens):
    """Return the normalized frequency of `term` in one document."""
    if not document_tokens:
        return 0.0
    return document_tokens.count(term) / len(document_tokens)


def document_frequency(term, corpus_tokens):
    """Return the number of documents containing `term` at least once."""
    return sum(term in set(document) for document in corpus_tokens)


def inverse_document_frequency(term, corpus_tokens):
    """Return log(N / DF) for a term in a non-empty corpus."""
    number_of_documents = len(corpus_tokens)
    if number_of_documents == 0:
        raise ValueError("The corpus cannot be empty.")

    df = document_frequency(term, corpus_tokens)
    if df == 0:
        raise ValueError(f"{term!r} does not occur in the corpus.")

    return math.log(number_of_documents / df)


def tf_idf(term, document_tokens, corpus_tokens):
    """Compute TF-IDF(term, document) = TF(term, document) * IDF(term)."""
    if term not in document_tokens:
        return 0.0
    return (
        term_frequency(term, document_tokens)
        * inverse_document_frequency(term, corpus_tokens)
    )


# Locate the repository when Jupyter starts from its root or a subfolder.
current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "phase-1-machine-learning-nlp").exists()
)

part_02 = repository_root / "phase-1-machine-learning-nlp" / "02-preprocessing-tokenization"
sys.path.insert(0, str(part_02 / "src"))
from preprocessor import preprocess_text

dataset_path = part_02 / "data" / "IMDB Dataset.csv"
reviews = pd.read_csv(dataset_path)

if "review" not in reviews.columns:
    raise KeyError("The IMDB dataset must contain a 'review' column.")

# Use the complete collection to calculate document frequency.
corpus_tokens = [
    preprocess_text(review, remove_stopwords=False, apply_stemming=False).split()
    for review in reviews["review"].dropna().astype(str)
]

# Compute one score for every unique word in the selected review.
review_number = 0
selected_review_tokens = corpus_tokens[review_number]
word_counts = Counter(selected_review_tokens)

tf_idf_rows = []
for word, count in word_counts.items():
    df = document_frequency(word, corpus_tokens)
    tf = term_frequency(word, selected_review_tokens)
    idf = inverse_document_frequency(word, corpus_tokens)
    tf_idf_rows.append(
        {
            "word": word,
            "count_in_review": count,
            "term_frequency": tf,
            "document_frequency": df,
            "inverse_document_frequency": idf,
            "tf_idf": tf_idf(word, selected_review_tokens, corpus_tokens),
        }
    )

tf_idf_results = (
    pd.DataFrame(tf_idf_rows)
    .sort_values(["tf_idf", "word"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Review {review_number} contains {len(selected_review_tokens)} words ")
print(f"and {len(word_counts)} unique words after preprocessing.")
tf_idf_results

Review 0 contains 300 words 
and 189 unique words after preprocessing.


,word,count_in_review,term_frequency,document_frequency,inverse_document_frequency,tf_idf
0,oz,5,0.016667,170,5.683980,0.094733
1,wholl,2,0.006667,33,7.323271,0.048822
2,violence,4,0.013333,1614,3.433307,0.045777
3,prison,3,0.010000,598,4.426188,0.044262
4,inmates,2,0.006667,76,6.489045,0.043260
...,...,...,...,...,...,...
184,of,7,0.023333,47409,0.053211,0.001242
185,this,3,0.010000,44907,0.107429,0.001074
186,and,6,0.020000,48208,0.036498,0.000730
187,the,15,0.050000,49537,0.009303,0.000465


## How does ranking work given any user query?

Ranking is the step that follows the retreival of all relevant documents (by returning the documents that have the highest average tf-idf (or other metrics) of the query terms) and then ranking them in descending order of relevance. Ranking can be by tf-idf aswell but this is not always the best way of doing it, the more modern way is a method that improves on the tf-idf and is called BM25 which handles things like term frequency saturation, the document length, and rare vs common terms. This could also have more conditions like do all terms in the query appear in the document, if yes are they close together, also could consider the document popularity and its quality. Considering all these things a score for each candidate document is calculated and the documents are ranked according to their score.

## How do most search engines work?

Search engines work in two phases, the first phase is on the document side:

1. On the document side, first document and webpages are collected by crawling mostly, then the text in these documents and webpages is parsed and extracted preprocessed and tokenized, finally the terms and the document index are added to the inverted index so they can be looked up when in the search phase.

2. The second phase is the search or query phase, this phase starts by a user entering a query, that query before searching for results is parsed and normalized, then every term in that query is lookedup in the inverted index and the documents that have those terms are returned as candidate documents, now the candidate documents are scored on their relevance to the user query using tf-idf and more advance methods like document popuarity and quality, also how many webpages point to it, then in most engines the last step is called filtering or access control in this step the engine makes sure that the user is allowed or even wants to see this document for example age-restricions or geographic region or language. After all this is done the results are given to the user.

In [9]:
from elasticsearch import Elasticsearch, helpers
import re


# Connect to an Elasticsearch server running locally.
elasticsearch_client = Elasticsearch("http://localhost:9200", request_timeout=30)

if not elasticsearch_client.ping():
    raise ConnectionError(
        "Could not connect to Elasticsearch at http://localhost:9200. "
        "Start the Elasticsearch server, then run this cell again."
    )

index_name = "imdb-reviews-search-demo"

# Create a dedicated index. The standard analyzer tokenizes and lowercases
# the review text, while the sentiment label is stored as an exact keyword.
if not elasticsearch_client.indices.exists(index=index_name):
    elasticsearch_client.indices.create(
        index=index_name,
        mappings={
            "properties": {
                "review": {"type": "text", "analyzer": "standard"},
                "sentiment": {"type": "keyword"},
            }
        },
    )

# Select the same documents on every run and keep their original row numbers
# as Elasticsearch IDs. Re-running the cell updates rather than duplicates them.
documents_to_index = (
    reviews.dropna(subset=["review", "sentiment"])
    .sample(n=min(100, len(reviews)), random_state=42)
)

indexing_actions = [
    {
        "_index": index_name,
        "_id": str(row_number),
        "_source": {
            "review": row["review"],
            "sentiment": row["sentiment"],
        },
    }
    for row_number, row in documents_to_index.iterrows()
]

indexed_count, indexing_errors = helpers.bulk(
    elasticsearch_client, indexing_actions, raise_on_error=False
)
elasticsearch_client.indices.refresh(index=index_name)

print(f"Successfully indexed {indexed_count} documents.")
if indexing_errors:
    print(f"Failed to index {len(indexing_errors)} documents.")

# A match query analyzes the user's text and ranks matching reviews by
# relevance. Change this string to try another query.
user_query = "great acting and story"
search_response = elasticsearch_client.search(
    index=index_name,
    query={"match": {"review": {"query": user_query}}},
    size=5,
)

hits = search_response["hits"]["hits"]
query_words = set(re.findall(r"\b\w+\b", user_query.lower()))


def uppercase_query_words(review, query_words):
    """Uppercase whole review words that occur in the query."""
    return re.sub(
        r"\b\w+\b",
        lambda match: (
            match.group(0).upper()
            if match.group(0).lower() in query_words
            else match.group(0)
        ),
        review,
    )
print(f"Top results for: {user_query!r}")
print(f"Query words shown in uppercase: {', '.join(sorted(query_words))}")
print(f"Number of results shown: {len(hits)}")

for rank, hit in enumerate(hits, start=1):
    print("\n" + "=" * 100)
    print(f"RESULT {rank}")
    print(f"Document ID: {hit['_id']}")
    print(f"Score: {hit['_score']}")
    print(f"Sentiment: {hit['_source']['sentiment']}")
    print("Review:")
    highlighted_review = uppercase_query_words(
        hit["_source"]["review"], query_words
    )
    print(highlighted_review)

if not hits:
    print("No matching reviews were found.")

Successfully indexed 100 documents.
Top results for: 'great acting and story'
Query words shown in uppercase: acting, and, great, story
Number of results shown: 5

RESULT 1
Document ID: 33109
Score: 5.241029
Sentiment: positive
Review:
Three kids are born during a solar eclipse AND turn into vile murderous little tykes who're above suspicion by everyone, save for Joyce (Lori Lethin) AND her younger brother Timmy. That's the STORY in a nutshell. The ACTING in this one is tolerable for the most part. Notable for MTV-J Julie Brown (not the 'Downtown' one) showing some skin, AND a very early part (albiet small) for Michael Dudikoff. Not a GREAT film by any stretch of the imagination, but in the 'killer kids' sub-genre it's a bit of a guilty pleasure.<br /><br />Eye Candy: Julie Brown shows T&A (the only film thus far, to claim that honor); Sylvia Wright gets topless <br /><br />DVD Extras (R1): 16 minute interview with Producer Max Rosenberg (wherein he insults the director AND Canada, GRE

In [11]:
# BM25 lexical search compares query terms with terms in each document.
# Semantic search instead compares embedding vectors that represent meaning.
# Cosine similarity measures how close the query meaning is to review meaning.
try:
    from sentence_transformers import SentenceTransformer
except ModuleNotFoundError as error:
    if error.name == "sentence_transformers":
        raise ModuleNotFoundError(
            "sentence-transformers is required for semantic search. "
            "Install it with: pip install sentence-transformers"
        ) from error
    raise


required_variables = [
    "elasticsearch_client",
    "documents_to_index",
    "user_query",
    "reviews",
]
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    raise RuntimeError(
        "Run the previous Elasticsearch cell first. Missing variables: "
        + ", ".join(missing_variables)
    )

if documents_to_index.empty:
    raise ValueError("documents_to_index contains no reviews to embed.")

server_info = elasticsearch_client.info()
server_version = server_info["version"]["number"]
server_major_version = int(server_version.split(".", maxsplit=1)[0])

semantic_index_name = "imdb-reviews-semantic-search-demo"
embedding_model_name = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_name, device="cpu")
embedding_dimensions = embedding_model.get_sentence_embedding_dimension()
if embedding_dimensions is None:
    raise ValueError("The embedding model did not report its vector dimensions.")

review_texts = documents_to_index["review"].astype(str).tolist()
review_embeddings = embedding_model.encode(
    review_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
if len(review_embeddings) != len(documents_to_index):
    raise RuntimeError("The number of embeddings does not match the documents.")

# Recreate only this dedicated demo index so reruns cannot create duplicates
# and its dense-vector mapping always matches the selected embedding model.
if elasticsearch_client.indices.exists(index=semantic_index_name):
    elasticsearch_client.indices.delete(index=semantic_index_name)

vector_mapping = {"type": "dense_vector", "dims": embedding_dimensions}
if server_major_version >= 8:
    vector_mapping.update({"index": True, "similarity": "cosine"})

elasticsearch_client.indices.create(
    index=semantic_index_name,
    mappings={
        "properties": {
            "review": {"type": "text"},
            "sentiment": {"type": "keyword"},
            "review_embedding": vector_mapping,
        }
    },
)

semantic_indexing_actions = [
    {
        "_index": semantic_index_name,
        "_id": str(row_number),
        "_source": {
            "review": str(row["review"]),
            "sentiment": row["sentiment"],
            "review_embedding": embedding.tolist(),
        },
    }
    for (row_number, row), embedding in zip(
        documents_to_index.iterrows(), review_embeddings
    )
]

semantic_indexed_count, semantic_indexing_errors = helpers.bulk(
    elasticsearch_client,
    semantic_indexing_actions,
    raise_on_error=False,
)
if semantic_indexing_errors:
    raise RuntimeError(
        f"Failed to index {len(semantic_indexing_errors)} semantic documents."
    )
if semantic_indexed_count != len(documents_to_index):
    raise RuntimeError(
        f"Indexed {semantic_indexed_count} of {len(documents_to_index)} documents."
    )
elasticsearch_client.indices.refresh(index=semantic_index_name)

query_embedding = embedding_model.encode(
    user_query,
    convert_to_numpy=True,
    normalize_embeddings=True,
).tolist()
result_count = min(5, len(documents_to_index))

# Elasticsearch 8+ supports native approximate k-nearest-neighbor search.
# Older dense-vector versions use a valid cosine script_score fallback.
if server_major_version >= 8:
    semantic_search_method = "native knn"
    semantic_response = elasticsearch_client.search(
        index=semantic_index_name,
        knn={
            "field": "review_embedding",
            "query_vector": query_embedding,
            "k": result_count,
            "num_candidates": min(100, len(documents_to_index)),
        },
        source=["review", "sentiment"],
        size=result_count,
    )
else:
    semantic_search_method = "cosine script_score"
    semantic_response = elasticsearch_client.search(
        index=semantic_index_name,
        query={
            "script_score": {
                "query": {"match_all": {}},
                "script": {
                    "source": (
                        "cosineSimilarity(params.query_vector, "
                        "'review_embedding') + 1.0"
                    ),
                    "params": {"query_vector": query_embedding},
                },
            }
        },
        source=["review", "sentiment"],
        size=result_count,
    )

semantic_hits = semantic_response["hits"]["hits"]
print(f"Semantic index: {semantic_index_name}")
print(f"Embedding model: {embedding_model_name}")
print(f"Elasticsearch server: {server_version} ({semantic_search_method})")
print(f"Query: {user_query!r}")
print(f"Number of returned results: {len(semantic_hits)}")

for rank, hit in enumerate(semantic_hits, start=1):
    # Elasticsearch shifts cosine scores to keep them non-negative.
    if semantic_search_method == "native knn":
        cosine_similarity = (2.0 * hit["_score"]) - 1.0
    else:
        cosine_similarity = hit["_score"] - 1.0

    print("\n" + "=" * 100)
    print(f"RESULT {rank}")
    print(f"Document ID: {hit['_id']}")
    print(f"Similarity score: {cosine_similarity:.6f}")
    print(f"Sentiment: {hit['_source']['sentiment']}")
    print("Review:")
    print(hit["_source"]["review"])

print(
    "\nThese results may differ from the previous BM25 results because semantic "
    "search compares meaning through embedding similarity instead of relying "
    "only on exact shared words."
)

/home/jadkandah/progressSoft_internship/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12546.86it/s]
/tmp/ipykernel_693813/31734779.py:38: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimensions = embedding_model.get_sentence_embedding_dimension()
Batches: 100%|██████████| 4/4 [00:02<00:00,  1.43it/s]


Semantic index: imdb-reviews-semantic-search-demo
Embedding model: all-MiniLM-L6-v2
Elasticsearch server: 9.4.1 (native knn)
Query: 'great acting and story'
Number of returned results: 5

RESULT 1
Document ID: 22062
Similarity score: 0.522190
Sentiment: positive
Review:
Though I can't claim to be a comic book fanatic, I have read my share, so I guess I'm part of the audience of this film, and I wasn't disappointed. It does run out of steam near the end, it's almost overflowing with ideas, and it seems like Lena Olin, one of my favorite actresses, was left on the cutting room floor. Also, a little of Hank Azaria's Blue Raja can go a long way. Still, it's easy to forgive all of these faults when you have a film which is this much fun. All the actors seem to be having a blast with their roles, especially William H. Macy as the straight-arrow Shoveler, and Janeane Garofalo as The Bowler. And unlike some, I found the design of the city to make the joke even funnier. I also liked how disco w